# z305 — Productos "mágicos" del profesor

Notebook **aislado** del pipeline z301-z304 (no lo modifica ni depende de él más que para leer el parquet ya generado).

Dos partes:
1. **Investigación**: ¿qué caracteriza a los 182 productos de la lista `productos_magicos` del notebook OLS del profe? (volumen, categoría, antigüedad, etc.)
2. **Experimento (opción B)**: entrenar LGBM restringido a esos 182 productos, pero usando **todo el histórico** (no solo 201812 como el OLS), para ver si restringir el universo de entrenamiento ayuda.

No sabemos a priori qué hace "mágicos" a estos productos — la investigación es puramente empírica.

## 0. Ambiente

In [ ]:
import os, shutil, subprocess

BASE = '/home/ds/buckets/b1'

kaggle_dst = os.path.expanduser('~/.kaggle/kaggle.json')
os.makedirs(os.path.dirname(kaggle_dst), exist_ok=True)
if os.path.exists(kaggle_dst):
    os.chmod(kaggle_dst, 0o600)
    print('Kaggle auth OK (ya estaba en ~/.kaggle)')
else:
    for cand in [f'{BASE}/kaggle.json', f'{BASE}/kaggle/kaggle.json']:
        if os.path.exists(cand):
            shutil.copy(cand, kaggle_dst)
            os.chmod(kaggle_dst, 0o600)
            print(f'Kaggle auth OK (copiado de {cand})')
            break
    else:
        print('⚠️  kaggle.json no encontrado')

In [ ]:
!pip install -q uv
!uv pip install -q polars pyarrow lightgbm kaggle scikit-learn optuna

In [ ]:
import polars as pl
import numpy as np
import lightgbm as lgb
import json
import warnings
warnings.filterwarnings('ignore')

PARAM = {
    'path_features':    '/home/ds/buckets/b1/exp/z302_features_producto.parquet',
    'path_inferencia':  '/home/ds/buckets/b1/exp/z302_inferencia_producto.parquet',
    'path_apredecir':   '/home/ds/buckets/b1/datasets/product_id_apredecir201912.txt',
    'kaggle_competition': 'labo-iii-2026-rosario',
    'semilla': 102191,
}

productos_magicos = [ 20001, 20002, 20005, 20013, 20033, 20037, 20038, 20043, 20044,
  20045, 20046, 20052, 20055, 20058, 20059, 20069, 20070, 20072, 20073, 20075, 20080,
  20091, 20094, 20099, 20107, 20114, 20120, 20132, 20137, 20139, 20142, 20144, 20146,
  20148, 20151, 20153, 20157, 20158, 20161, 20162, 20166, 20167, 20189, 20198, 20201,
  20202, 20203, 20208, 20226, 20228, 20231, 20233, 20253, 20254, 20256, 20269, 20270,
  20271, 20275, 20276, 20277, 20278, 20288, 20298, 20315, 20317, 20320, 20322, 20335,
  20337, 20338, 20344, 20348, 20350, 20353, 20359, 20385, 20390, 20398, 20402, 20403,
  20406, 20411, 20416, 20417, 20418, 20419, 20421, 20422, 20424, 20428, 20429, 20443,
  20456, 20466, 20469, 20479, 20497, 20500, 20509, 20514, 20517, 20524, 20532, 20549,
  20551, 20560, 20561, 20565, 20568, 20579, 20583, 20585, 20586, 20589, 20599, 20606,
  20614, 20624, 20632, 20642, 20646, 20653, 20655, 20657, 20660, 20661, 20663, 20666,
  20677, 20680, 20684, 20696, 20699, 20713, 20737, 20744, 20745, 20765, 20768, 20773,
  20777, 20786, 20789, 20800, 20807, 20812, 20818, 20830, 20832, 20838, 20847, 20855,
  20863, 20864, 20882, 20883, 20906, 20913, 20914, 20919, 20922, 20925, 20937, 20945,
  20956, 20961, 20965, 20970, 20976, 20986, 20996, 21016, 21038, 21048, 21049, 21077,
  21080, 21088, 21118, 21170, 21200
]
print(f'Productos mágicos: {len(productos_magicos)} (sin duplicados: {len(set(productos_magicos))})')

## PARTE 1 — ¿Qué caracteriza a los productos mágicos?

Comparamos volumen, categoría, antigüedad e intermitencia de los 182 mágicos contra el resto de los 780. Es una exploración empírica: no sabemos a priori el criterio del profesor.

In [ ]:
df = pl.read_parquet(PARAM['path_features'])
print(f'Dataset: {df.shape}')

# Un producto es "mágico" o no (columna auxiliar, no se guarda en el dataset real)
df_prod = df.select(['product_id']).unique()
es_magico = df_prod['product_id'].is_in(productos_magicos)
print(f'Productos en el dataset que son mágicos: {es_magico.sum()} de {len(productos_magicos)} en la lista')
# Si no coinciden los 182, algunos mágicos no están en el universo de 780 o el dataset

In [ ]:
# Caracterización por producto: volumen total, meses activo, % ceros, categoría
caracteristicas = (
    df.group_by('product_id')
      .agg([
          pl.col('tn').sum().alias('tn_total'),
          pl.col('tn').mean().alias('tn_media'),
          pl.len().alias('meses_en_dataset'),
          (pl.col('tn') == 0).mean().alias('pct_ceros'),
          pl.col('cat1').first().alias('cat1'),
          pl.col('cat2').first().alias('cat2'),
          pl.col('cat3').first().alias('cat3'),
          pl.col('brand').first().alias('brand'),
      ])
)
caracteristicas = caracteristicas.with_columns(
    pl.col('product_id').is_in(productos_magicos).alias('es_magico')
)

print('── Comparación mágicos vs resto ──')
resumen = caracteristicas.group_by('es_magico').agg([
    pl.len().alias('n'),
    pl.col('tn_total').mean().alias('tn_total_prom'),
    pl.col('tn_total').median().alias('tn_total_mediana'),
    pl.col('tn_media').mean().alias('tn_media_prom'),
    pl.col('pct_ceros').mean().alias('pct_ceros_prom'),
    pl.col('meses_en_dataset').mean().alias('meses_prom'),
])
print(resumen)

# ¿Los mágicos son los de mayor volumen? Percentil de tn_total dentro del universo completo
caracteristicas = caracteristicas.with_columns(
    pl.col('tn_total').rank(descending=True).alias('rank_volumen')
)
print('\n── Ranking de volumen de los mágicos (1=más grande de los 780) ──')
print(caracteristicas.filter(pl.col('es_magico')).select(['product_id','tn_total','rank_volumen']).sort('rank_volumen').head(10))
print('...')
print(caracteristicas.filter(pl.col('es_magico')).select(['product_id','tn_total','rank_volumen']).sort('rank_volumen').tail(10))

In [ ]:
# ¿Qué fracción del WAPE total (peso de volumen) representan los mágicos?
tn_total_780 = caracteristicas['tn_total'].sum()
tn_total_magicos = caracteristicas.filter(pl.col('es_magico'))['tn_total'].sum()
print(f'Tn total de los 780: {tn_total_780:,.0f}')
print(f'Tn total de los 182 mágicos: {tn_total_magicos:,.0f}')
print(f'Representan el {100*tn_total_magicos/tn_total_780:.1f}% del volumen total (y por lo tanto del peso en el WAPE)')
print(f'({len(productos_magicos)} de 780 = {100*len(productos_magicos)/780:.1f}% de los productos)')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución de volumen: mágicos vs resto
for label, grupo in [('Mágicos', True), ('Resto', False)]:
    datos = caracteristicas.filter(pl.col('es_magico') == grupo)['tn_total'].to_numpy()
    axes[0].hist(np.log1p(datos), bins=30, alpha=0.5, label=label, density=True)
axes[0].set_title('log(1+tn_total) por grupo')
axes[0].legend()

# % de ceros
for label, grupo in [('Mágicos', True), ('Resto', False)]:
    datos = caracteristicas.filter(pl.col('es_magico') == grupo)['pct_ceros'].to_numpy()
    axes[1].hist(datos, bins=20, alpha=0.5, label=label, density=True)
axes[1].set_title('% de meses en cero por grupo')
axes[1].legend()

# Distribución por cat1
cat_counts = caracteristicas.group_by(['cat1', 'es_magico']).len().sort('cat1')
cat_pivot = cat_counts.pivot(values='len', index='cat1', on='es_magico').fill_null(0)
print('\nDistribución por cat1 (mágicos vs resto):')
print(cat_pivot)

plt.tight_layout()
plt.show()

## PARTE 2 — Opción B: LGBM restringido a los productos mágicos (todo el histórico)

A diferencia del OLS del profe (que entrena solo con 201812), acá usamos **todo el histórico disponible**, pero solo con las filas de los 182 productos mágicos. La hipótesis: si esos productos son especiales (por volumen, estabilidad, etc.), un modelo enfocado en ellos podría predecir mejor esa porción, aunque haya que decidir qué hacer con el resto de los 780 para el submit completo.

In [ ]:
# Filtrar el dataset de train a solo los productos mágicos, TODO el histórico
df_train_full = pl.read_parquet(PARAM['path_features'])
df_infer_full = pl.read_parquet(PARAM['path_inferencia'])

df_train_mag = df_train_full.filter(pl.col('product_id').is_in(productos_magicos))
print(f'Train solo mágicos, todo el histórico: {df_train_mag.height:,} filas (vs {df_train_full.height:,} filas totales)')

# Target: nivel (igual que tu mejor config)
TARGET_COL = 'target_nivel'
COLS_EXCLUIR_BASE = [
    'agrupa_id', 'product_id', 'customer_id', 'periodo',
    'tn', 'tn_t2', 'target_nivel', 'target_delta',
    'modo_agrupacion', 'solo_predecir', 'tipo_target',
]
FEATURES = [c for c in df_train_mag.columns if c not in COLS_EXCLUIR_BASE]
COLS_CATEGORICAS = ['cat1', 'cat2', 'cat3', 'brand']
CAT_FEATURES = [c for c in COLS_CATEGORICAS if c in FEATURES]
print(f'Features: {len(FEATURES)}  |  Categóricas: {CAT_FEATURES}')

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

df_pd = df_train_mag.to_pandas()
for c in CAT_FEATURES:
    df_pd[c] = df_pd[c].astype('category')

# Split de validación: último período con target conocido dentro de los mágicos
periodos_mag = sorted(df_pd['periodo'].unique())
val_p = periodos_mag[-1]
print(f'Validando en periodo {val_p}  (train: todo lo anterior)')

def calcular_wape(real, pred):
    real = np.asarray(real); pred = np.asarray(pred)
    return np.abs(real - pred).sum() / real.sum()

def objective(trial):
    params = {
        'objective': 'tweedie',
        'tweedie_variance_power': trial.suggest_float('tweedie_variance_power', 1.1, 1.9),
        'metric': 'mae',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'seed': PARAM['semilla'],
        'num_leaves': trial.suggest_int('num_leaves', 8, 128),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.2, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    df_tr = df_pd[df_pd['periodo'] < val_p]
    df_vl = df_pd[df_pd['periodo'] == val_p]

    modelo_trial = lgb.LGBMRegressor(**params)
    modelo_trial.fit(
        df_tr[FEATURES], df_tr[TARGET_COL],
        eval_set=[(df_vl[FEATURES], df_vl[TARGET_COL])],
        categorical_feature=CAT_FEATURES,
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )
    pred = np.maximum(modelo_trial.predict(df_vl[FEATURES]), 0.0)
    return calcular_wape(df_vl[TARGET_COL].values, pred)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f'\nMejor WAPE (val {val_p}): {study.best_value:.4f}')
print('Mejores hiperparámetros:', study.best_params)

# Modelo FINAL: reentrena con TODOS los datos de los mágicos (incluido val_p)
# usando los mejores hiperparámetros encontrados
best_params = {
    'objective': 'tweedie',
    'metric': 'mae',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'seed': PARAM['semilla'],
    **study.best_params,
}
X_train = df_pd[FEATURES]
y_train = df_pd[TARGET_COL]
modelo = lgb.LGBMRegressor(**best_params)
modelo.fit(X_train, y_train, categorical_feature=CAT_FEATURES)
print('✅ Modelo final entrenado sobre TODOS los mágicos con hiperparámetros optimizados.')

## 3. Predicción

Este modelo solo vio productos mágicos. Para el submit completo (780 productos) hay dos caminos:
- **(a)** usarlo solo para los mágicos, y completar el resto con tu mejor modelo global (0.249) — un ensamble por segmento
- **(b)** aplicarlo a los 780 igual, para ver qué tan bien generaliza fuera de su universo de entrenamiento (probablemente mal, pero es informativo)

Acá hacemos **(a)**, que es el uso más razonable de este experimento.

In [ ]:
# Filtramos a periodo == 201912: es la única fila que predice febrero 2020 (horizonte=2).
# df_infer_full trae 2 filas por producto (201911 y 201912, ambas con target nulo);
# sin este filtro se duplican las predicciones.
df_infer_pd = df_infer_full.filter(pl.col('periodo') == 201912).to_pandas()
for c in CAT_FEATURES:
    df_infer_pd[c] = df_infer_pd[c].astype('category')

X_infer = df_infer_pd[FEATURES]
pred = modelo.predict(X_infer)
pred = np.maximum(pred, 0.0)

df_pred_magicos = pl.DataFrame({
    'product_id': df_infer_pd['product_id'].values,
    'tn_magico': pred,
}).filter(pl.col('product_id').is_in(productos_magicos))

print(f'Predicciones para mágicos: {df_pred_magicos.height} productos (debería ser 182)')
print(df_pred_magicos.describe())

## 4. Combinar con el modelo global (0.249) para el resto

Si tenés un CSV de submit ya generado por z304 (tu mejor modelo, 0.249), lo cargamos y reemplazamos las predicciones de los productos mágicos por las de este modelo especializado.

In [ ]:
# Ajustá esta ruta al submit de tu mejor modelo (z304) que quieras usar como base
path_submit_base = '/home/ds/buckets/b1/exp/z304_predicciones_producto.csv'

df_base = pl.read_csv(path_submit_base)
print(f'Submit base: {df_base.height} productos')
print(df_base.columns)

In [ ]:
# Combinar: reemplazar tn de los mágicos por la predicción del modelo especializado
col_tn = 'tn' if 'tn' in df_base.columns else df_base.columns[-1]

df_final = (
    df_base
    .join(df_pred_magicos, on='product_id', how='left')
    .with_columns(pl.coalesce([pl.col('tn_magico'), pl.col(col_tn)]).alias(col_tn))
    .drop('tn_magico')
)

print(f'Submit combinado: {df_final.height} productos')
print(f'  Mágicos reemplazados: {df_pred_magicos.height}')
print(f'  Resto sin cambios: {df_final.height - df_pred_magicos.height}')

archivo = '/home/ds/buckets/b1/exp/z305_combinado_magicos.csv'
df_final.write_csv(archivo)
print(f'✅ Guardado: {archivo}')

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    import subprocess
    res = subprocess.run(
        ['kaggle', 'competitions', 'submit', '-c', competencia, '-f', archivo, '-m', mensaje],
        capture_output=True, text=True
    )
    print('stdout:', res.stdout)
    print('stderr:', res.stderr)

mensaje = f"z305 | LGBM productos_magicos (todo histórico) + resto modelo global 0.249"
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)

## PARTE 5 — Aplicar los hiperparámetros de Optuna (buscados en mágicos) a los 780

Se reusan los `best_params` encontrados en la Parte 2 (optimizados sobre el subconjunto de 182 mágicos), pero el modelo final se entrena sobre **todos los 780 productos**. Hipótesis: esos hiperparámetros podrían generalizar bien al dataset completo, sin correr Optuna de nuevo (que sería lo que ya hace z303).

No hay garantía de que mejore — los hiperparámetros óptimos para un subconjunto chico no necesariamente son óptimos para el dataset completo.

In [ ]:
# Reusa study.best_params de la Parte 2 (Optuna corrido sobre los 182 mágicos)
df_pd_full = df_train_full.to_pandas()
for c in CAT_FEATURES:
    df_pd_full[c] = df_pd_full[c].astype('category')

X_train_full = df_pd_full[FEATURES]
y_train_full = df_pd_full[TARGET_COL]

best_params_780 = {
    'objective': 'tweedie',
    'metric': 'mae',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'seed': PARAM['semilla'],
    **study.best_params,   # hiperparámetros encontrados sobre los mágicos
}

modelo_780 = lgb.LGBMRegressor(**best_params_780)
modelo_780.fit(X_train_full, y_train_full, categorical_feature=CAT_FEATURES)
print('✅ Modelo entrenado sobre los 780 con hiperparámetros de mágicos.')
print('Hiperparámetros usados:', study.best_params)

### Predicción y submit

In [ ]:
df_infer_pd_full = df_infer_full.filter(pl.col('periodo') == 201912).to_pandas()
for c in CAT_FEATURES:
    df_infer_pd_full[c] = df_infer_pd_full[c].astype('category')

X_infer_full = df_infer_pd_full[FEATURES]
pred_780 = np.maximum(modelo_780.predict(X_infer_full), 0.0)

df_submit_780 = pl.DataFrame({
    'product_id': df_infer_pd_full['product_id'].values,
    'tn': pred_780,
})
print(f'Predicciones: {df_submit_780.height} productos (debería ser 780)')
print(df_submit_780.describe())

archivo_780 = '/home/ds/buckets/b1/exp/z305_hiperparams_magicos_en_780.csv'
df_submit_780.write_csv(archivo_780)
print(f'✅ Guardado: {archivo_780}')

mensaje_780 = f"z305 | LGBM 780 productos con hiperparámetros optimizados en mágicos"
kaggle_submit(PARAM['kaggle_competition'], archivo_780, mensaje_780)

## PARTE 2b — Reproducir con hiperparámetros fijos (diagnóstico)

Entrena de nuevo sobre los 182 mágicos, pero con los hiperparámetros exactos que Optuna encontró en la corrida que dio 0.252 (fijos, sin volver a correr Optuna). El objetivo es aislar si la discrepancia (0.252 vs 0.263) vino de la variabilidad de Optuna o de otra causa en el pipeline de combinación.

Si este submit reproduce 0.252, confirma que el problema anterior fue variabilidad de Optuna (o el bug de duplicados en alguna corrida intermedia). Si NO lo reproduce, hay algo más que investigar.

In [ ]:
best_params_fijos = {
    'objective': 'tweedie',
    'metric': 'mae',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'seed': PARAM['semilla'],
    'tweedie_variance_power': 1.7284238248235682,
    'num_leaves': 27,
    'max_depth': 7,
    'learning_rate': 0.0823053685706306,
    'n_estimators': 870,
    'min_child_samples': 57,
    'subsample': 0.5090255541322626,
    'colsample_bytree': 0.5303501783218144,
    'reg_alpha': 0.006630062029034843,
    'reg_lambda': 0.007145653270713575,
}

X_train = df_pd[FEATURES]
y_train = df_pd[TARGET_COL]

modelo = lgb.LGBMRegressor(**best_params_fijos)
modelo.fit(X_train, y_train, categorical_feature=CAT_FEATURES)
print('✅ Modelo entrenado sobre los 182 mágicos con hiperparámetros FIJOS (reproducción).')
print('Recordatorio: correr después parte3_predict y parte4_combine con este `modelo`.')

## PARTE 5b — Los mismos hiperparámetros fijos, entrenados sobre los 780

A diferencia de la Parte 5 (que usa `study.best_params` de la corrida más reciente de Optuna, variable), acá se usan los hiperparámetros FIJOS de arriba, entrenados sobre el dataset completo de 780 productos.

In [ ]:
df_pd_full = df_train_full.to_pandas()
for c in CAT_FEATURES:
    df_pd_full[c] = df_pd_full[c].astype('category')

X_train_full = df_pd_full[FEATURES]
y_train_full = df_pd_full[TARGET_COL]

modelo_780_fijo = lgb.LGBMRegressor(**best_params_fijos)
modelo_780_fijo.fit(X_train_full, y_train_full, categorical_feature=CAT_FEATURES)
print('✅ Modelo entrenado sobre los 780 con hiperparámetros FIJOS (los que dieron 0.252 en mágicos).')

In [ ]:
df_infer_pd_full = df_infer_full.filter(pl.col('periodo') == 201912).to_pandas()
for c in CAT_FEATURES:
    df_infer_pd_full[c] = df_infer_pd_full[c].astype('category')

X_infer_full = df_infer_pd_full[FEATURES]
pred_780_fijo = np.maximum(modelo_780_fijo.predict(X_infer_full), 0.0)

df_submit_780_fijo = pl.DataFrame({
    'product_id': df_infer_pd_full['product_id'].values,
    'tn': pred_780_fijo,
})
print(f'Predicciones: {df_submit_780_fijo.height} productos (debería ser 780)')
print(df_submit_780_fijo.describe())

archivo_780_fijo = '/home/ds/buckets/b1/exp/z305_hiperparams_fijos_en_780.csv'
df_submit_780_fijo.write_csv(archivo_780_fijo)
print(f'✅ Guardado: {archivo_780_fijo}')

mensaje_780_fijo = "z305 | LGBM 780 con hiperparámetros FIJOS (los que dieron 0.252 en mágicos)"
kaggle_submit(PARAM['kaggle_competition'], archivo_780_fijo, mensaje_780_fijo)

## PARTE 5c — Ensemble de semillas sobre los 780 (hiperparámetros fijos)

Entrena varios modelos con los mismos hiperparámetros fijos (los que dieron 0.247), variando solo la semilla de LightGBM, y promedia las predicciones. Sirve para ver si 0.247 es estable o varía por semilla, y si el ensemble mejora aún más.

In [ ]:
semillas_ensemble = [102191, 102193, 102197, 102199, 102203]

modelos_ensemble = []
preds_individuales = {}

for s in semillas_ensemble:
    params_s = dict(best_params_fijos)
    params_s['seed'] = s
    m = lgb.LGBMRegressor(**params_s)
    m.fit(X_train_full, y_train_full, categorical_feature=CAT_FEATURES)
    modelos_ensemble.append(m)

    pred_s = np.maximum(m.predict(X_infer_full), 0.0)
    preds_individuales[s] = pred_s
    print(f'  ✅ semilla {s} entrenada  (media pred={pred_s.mean():.2f})')

# Promedio del ensemble
pred_ensemble = np.mean(list(preds_individuales.values()), axis=0)

# Cuánto varían las predicciones individuales entre semillas (por producto)
std_entre_semillas = np.std(list(preds_individuales.values()), axis=0)
print(f'\nVariación entre semillas (std promedio por producto): {std_entre_semillas.mean():.3f}')
print(f'Como % de la predicción promedio: {100*std_entre_semillas.mean()/pred_ensemble.mean():.1f}%')

### Submit del ensemble

In [ ]:
df_submit_ensemble = pl.DataFrame({
    'product_id': df_infer_pd_full['product_id'].values,
    'tn': pred_ensemble,
})
print(df_submit_ensemble.describe())

archivo_ensemble = '/home/ds/buckets/b1/exp/z305_ensemble_5semillas_780.csv'
df_submit_ensemble.write_csv(archivo_ensemble)
print(f'✅ Guardado: {archivo_ensemble}')

mensaje_ensemble = f"z305 | LGBM 780 hiperparams fijos | ensemble {len(semillas_ensemble)} semillas"
kaggle_submit(PARAM['kaggle_competition'], archivo_ensemble, mensaje_ensemble)

## PARTE 5d — Ensemble con OTRO conjunto de semillas

Mismos hiperparámetros fijos, pero un conjunto de semillas distinto al anterior (que incluía la 102191). Objetivo: ver si el resultado converge cerca de 0.247 o si vuelve a caer en la zona 0.25-0.26, para saber si la semilla 102191 tenía una ventaja específica en el leaderboard público.

In [ ]:
# Conjunto de semillas SIN la 102191, para aislar su efecto
semillas_ensemble_2 = [7, 42, 777, 2024, 55555]

modelos_ensemble_2 = []
preds_individuales_2 = {}

for s in semillas_ensemble_2:
    params_s = dict(best_params_fijos)
    params_s['seed'] = s
    m = lgb.LGBMRegressor(**params_s)
    m.fit(X_train_full, y_train_full, categorical_feature=CAT_FEATURES)
    modelos_ensemble_2.append(m)

    pred_s = np.maximum(m.predict(X_infer_full), 0.0)
    preds_individuales_2[s] = pred_s
    print(f'  ✅ semilla {s} entrenada  (media pred={pred_s.mean():.2f})')

pred_ensemble_2 = np.mean(list(preds_individuales_2.values()), axis=0)

std_entre_semillas_2 = np.std(list(preds_individuales_2.values()), axis=0)
print(f'\nVariación entre semillas (std promedio por producto): {std_entre_semillas_2.mean():.3f}')
print(f'Como % de la predicción promedio: {100*std_entre_semillas_2.mean()/pred_ensemble_2.mean():.1f}%')

### Submit del segundo ensemble

In [ ]:
df_submit_ensemble_2 = pl.DataFrame({
    'product_id': df_infer_pd_full['product_id'].values,
    'tn': pred_ensemble_2,
})
print(df_submit_ensemble_2.describe())

archivo_ensemble_2 = '/home/ds/buckets/b1/exp/z305_ensemble_5semillas_780_v2.csv'
df_submit_ensemble_2.write_csv(archivo_ensemble_2)
print(f'✅ Guardado: {archivo_ensemble_2}')

mensaje_ensemble_2 = f"z305 | LGBM 780 hiperparams fijos | ensemble 5 semillas (set 2, sin 102191)"
kaggle_submit(PARAM['kaggle_competition'], archivo_ensemble_2, mensaje_ensemble_2)